# 06 — Commercial Density

Computes shop-specific density and diversity metrics per grid cell from OSM.

**Data source:** Overpass API (OSM `shop=*`) — fully portable.

**Method:** Single batch Overpass query for all shops, then BallTree matching to nearest grid cell.

**Output columns:** `cell_id`, `shop_density_km2`, `brand_ratio`

**Output file:** `csv/Houston/06_commercial_density.csv`

In [13]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [14]:
import pandas as pd
import numpy as np
import requests
import json
import os
import math
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("cache", exist_ok=True)

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
CELL_AREA_KM2 = (CELL_SIZE_M / 1000) ** 2
QUERY_RADIUS = 500
CSV_DIR = config.get("csv_dir", "csv")

os.makedirs(CSV_DIR, exist_ok=True)

df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
print(f"Loaded {len(df_grid)} grid cells")

BUFFER = 0.015
LAT_MIN = df_grid["cell_lat"].min() - BUFFER
LAT_MAX = df_grid["cell_lat"].max() + BUFFER
LON_MIN = df_grid["cell_lon"].min() - BUFFER
LON_MAX = df_grid["cell_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

Loaded 54947 grid cells


In [15]:
# ── Overpass batch query helper ───────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "grid-finding/1.0 (research project)"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            import time
            time.sleep(5 + attempt * 3)
    raise RuntimeError(f"Overpass failed: {last_error}")


print("Helpers ready.")

Helpers ready.


In [16]:
# ── Batch query: ALL shops in study area ──────────────

query = (f'[out:json][timeout:90];\n'
         f'(node["shop"]({BBOX});\n'
         f' way["shop"]({BBOX}););\n'
         f'out center tags;')

print("Querying all shops...")
data = query_overpass_cached(query)

EARTH_RADIUS_M = 6371000
MAX_DIST_M = QUERY_RADIUS

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    shop_val = tags.get("shop", "")
    if shop_val and shop_val not in {"vacant", "yes"}:
        lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
        lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
        if lat and lon:
            poi_records.append({
                "lat": float(lat), "lon": float(lon),
                "shop_type": shop_val,
                "has_brand": bool(tags.get("brand")),
            })

print(f"  Found {len(poi_records)} shops")

Querying all shops...
  Found 9735 shops


In [17]:
# ── BallTree: assign each shop to nearest grid cell ──

cell_coords_rad = np.radians(df_grid[["cell_lat", "cell_lon"]].values)
cell_ids = df_grid["cell_id"].tolist()

cell_shop_count = {c: 0 for c in cell_ids}
cell_brand_count = {c: 0 for c in cell_ids}

if poi_records:
    cell_tree = BallTree(cell_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = cell_tree.query(poi_coords, k=1)

    for j, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            cid = cell_ids[idx]
            cell_shop_count[cid] += 1
            if poi_records[j]["has_brand"]:
                cell_brand_count[cid] += 1

records = []
for cid in cell_ids:
    total_shops = cell_shop_count[cid]
    records.append({
        "cell_id": cid,
        "shop_density_km2": round(total_shops / CELL_AREA_KM2, 2),
        "brand_ratio": round(cell_brand_count[cid] / total_shops, 4) if total_shops > 0 else 0.0,
    })

df_shops = pd.DataFrame(records)
print(f"Completed: {len(df_shops)} cells")
print(f"Mean shops per cell: {(df_shops['shop_density_km2'] * CELL_AREA_KM2).mean():.1f}")
print(f"Cells with zero shops: {(df_shops['shop_density_km2'] == 0).sum()}")

Completed: 54947 cells
Mean shops per cell: 0.1
Cells with zero shops: 52029


In [18]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/06_commercial_density.csv"
df_shops.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_shops)} rows x {df_shops.shape[1]} cols)")
print(df_shops.describe().round(2).to_string())

Saved: csv/Houston/06_commercial_density.csv  (54947 rows x 3 cols)
       shop_density_km2  brand_ratio
count          54947.00     54947.00
mean               5.95         0.02
std               44.73         0.13
min                0.00         0.00
25%                0.00         0.00
50%                0.00         0.00
75%                0.00         0.00
max             3200.00         1.00


# 06 — Commercial Density

Computes shop-specific density and diversity metrics per grid cell from OSM.

**Data source:** Overpass API (OSM `shop=*`) — fully portable.

**Method:** Single batch Overpass query for all shops, then BallTree matching to nearest grid cell.

**Output columns:** `cell_id`, `shop_density_km2`, `brand_ratio`

**Output file:** `csv/06_commercial_density.csv`

In [19]:
# ── Papermill parameters ──────────────────────────────
GRID_CONFIG = "grid.json"

In [20]:
import pandas as pd
import numpy as np
import requests
import json
import os
import math
import hashlib
from sklearn.neighbors import BallTree

os.makedirs("cache", exist_ok=True)

with open(GRID_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

CELL_SIZE_M = config["grid_cell_size_m"]
CELL_AREA_KM2 = (CELL_SIZE_M / 1000) ** 2
QUERY_RADIUS = 500
CSV_DIR = config.get("csv_dir", "csv")

os.makedirs(CSV_DIR, exist_ok=True)

df_grid = pd.read_csv(f"{CSV_DIR}/01_grid_definition.csv", dtype={"cell_id": str})
print(f"Loaded {len(df_grid)} grid cells")

BUFFER = 0.015
LAT_MIN = df_grid["cell_lat"].min() - BUFFER
LAT_MAX = df_grid["cell_lat"].max() + BUFFER
LON_MIN = df_grid["cell_lon"].min() - BUFFER
LON_MAX = df_grid["cell_lon"].max() + BUFFER
BBOX = f"{LAT_MIN},{LON_MIN},{LAT_MAX},{LON_MAX}"

Loaded 54947 grid cells


In [21]:
# ── Overpass batch query helper ───────────────────────

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "grid-finding/1.0 (research project)"}


def _cache_path(query):
    h = hashlib.sha1(query.encode()).hexdigest()
    return f"cache/{h}.json"


def query_overpass_cached(query, max_retries=3):
    cp = _cache_path(query)
    if os.path.exists(cp):
        with open(cp, encoding="utf-8") as f:
            return json.load(f)
    last_error = None
    for attempt in range(max_retries):
        ep = OVERPASS_ENDPOINTS[attempt % len(OVERPASS_ENDPOINTS)]
        try:
            r = requests.post(ep, data={"data": query}, headers=HEADERS, timeout=120)
            r.raise_for_status()
            data = r.json()
            with open(cp, "w", encoding="utf-8") as f:
                json.dump(data, f)
            return data
        except Exception as e:
            last_error = e
            import time
            time.sleep(5 + attempt * 3)
    raise RuntimeError(f"Overpass failed: {last_error}")


print("Helpers ready.")

Helpers ready.


In [22]:
# ── Batch query: ALL shops in study area ──────────────

query = (f'[out:json][timeout:90];\n'
         f'(node["shop"]({BBOX});\n'
         f' way["shop"]({BBOX}););\n'
         f'out center tags;')

print("Querying all shops...")
data = query_overpass_cached(query)

EARTH_RADIUS_M = 6371000
MAX_DIST_M = QUERY_RADIUS

poi_records = []
for el in data.get("elements", []):
    tags = el.get("tags", {})
    shop_val = tags.get("shop", "")
    if shop_val and shop_val not in {"vacant", "yes"}:
        lat = el.get("lat") or (el.get("center", {}) or {}).get("lat")
        lon = el.get("lon") or (el.get("center", {}) or {}).get("lon")
        if lat and lon:
            poi_records.append({
                "lat": float(lat), "lon": float(lon),
                "shop_type": shop_val,
                "has_brand": bool(tags.get("brand")),
            })

print(f"  Found {len(poi_records)} shops")

Querying all shops...
  Found 9735 shops


In [23]:
# ── BallTree: assign each shop to nearest grid cell ──

cell_coords_rad = np.radians(df_grid[["cell_lat", "cell_lon"]].values)
cell_ids = df_grid["cell_id"].tolist()

cell_shop_count = {c: 0 for c in cell_ids}
cell_brand_count = {c: 0 for c in cell_ids}

if poi_records:
    cell_tree = BallTree(cell_coords_rad, metric="haversine")
    poi_coords = np.radians([[p["lat"], p["lon"]] for p in poi_records])
    distances, indices = cell_tree.query(poi_coords, k=1)

    for j, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
        if dist * EARTH_RADIUS_M <= MAX_DIST_M:
            cid = cell_ids[idx]
            cell_shop_count[cid] += 1
            if poi_records[j]["has_brand"]:
                cell_brand_count[cid] += 1

records = []
for cid in cell_ids:
    total_shops = cell_shop_count[cid]
    records.append({
        "cell_id": cid,
        "shop_density_km2": round(total_shops / CELL_AREA_KM2, 2),
        "brand_ratio": round(cell_brand_count[cid] / total_shops, 4) if total_shops > 0 else 0.0,
    })

df_shops = pd.DataFrame(records)
print(f"Completed: {len(df_shops)} cells")
print(f"Mean shops per cell: {(df_shops['shop_density_km2'] * CELL_AREA_KM2).mean():.1f}")
print(f"Cells with zero shops: {(df_shops['shop_density_km2'] == 0).sum()}")

Completed: 54947 cells
Mean shops per cell: 0.1
Cells with zero shops: 52029


In [26]:
# ── Save output ───────────────────────────────────────
output_path = f"{CSV_DIR}/06_commercial_density.csv"
df_shops.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_shops)} rows x {df_shops.shape[1]} cols)")
print(df_shops.describe().round(2).to_string())

Saved: csv/Houston/06_commercial_density.csv  (54947 rows x 3 cols)
       shop_density_km2  brand_ratio
count          54947.00     54947.00
mean               5.95         0.02
std               44.73         0.13
min                0.00         0.00
25%                0.00         0.00
50%                0.00         0.00
75%                0.00         0.00
max             3200.00         1.00
